# Validación y Dataset Maestro

**Fase:** Ingeniería del Dato 

---

En este notebook creo el **dataset maestro** uniendo los precios diarios de las 35 empresas del IBEX 35 con todas las variables macroeconómicas, alineando las distintas frecuencias temporales (diaria, mensual y trimestral).

## 1. Importaciones y rutas

Importo las librerías que necesitaré en el cuaderno y ruteo a los archivos

In [2]:
import pandas as pd
import numpy as np
import os
import sqlite3
from sqlalchemy import create_engine, Date
import warnings
warnings.filterwarnings('ignore')

BASE     = os.path.expanduser('~/Library/Mobile Documents/com~apple~CloudDocs/UFV/UNIVERSIDAD FRANCISCO DE VITORIA/4º/TFG')
DB_PATH  = os.path.join(BASE, 'proyecto', 'data', 'db', 'tfg.db')
PROC_DIR = os.path.join(BASE, 'proyecto', 'data', 'processed')
engine   = create_engine(f'sqlite:///{DB_PATH}')

print(f'DB: {DB_PATH}')

DB: /Users/adriancelada/Library/Mobile Documents/com~apple~CloudDocs/UFV/UNIVERSIDAD FRANCISCO DE VITORIA/4º/TFG/proyecto/data/db/tfg.db


## 2. Carga de tablas desde SQLite

Cargo las tablas desde `SQLite` y reviso la longitud de cada una

In [3]:
with sqlite3.connect(DB_PATH) as conn:
    df_precios    = pd.read_sql('SELECT * FROM precios_empresas', conn, parse_dates=['fecha'])
    df_act_mens   = pd.read_sql('SELECT * FROM macro_act_mensual', conn, parse_dates=['fecha'])
    df_act_trim   = pd.read_sql('SELECT * FROM macro_act_trimes', conn, parse_dates=['fecha'])
    df_mon_diario = pd.read_sql('SELECT * FROM macro_mon_diario', conn, parse_dates=['fecha'])
    df_mon_mens   = pd.read_sql('SELECT * FROM macro_mon_mensual', conn, parse_dates=['fecha'])
    df_riesgo     = pd.read_sql('SELECT * FROM macro_riesgo', conn, parse_dates=['fecha'])

print(f'precios_empresas : {len(df_precios):>7,} filas | {df_precios["ticker"].nunique()} empresas')
print(f'macro_act_mensual: {len(df_act_mens):>7,} filas')
print(f'macro_act_trimes : {len(df_act_trim):>7,} filas')
print(f'macro_mon_diario : {len(df_mon_diario):>7,} filas')
print(f'macro_mon_mensual: {len(df_mon_mens):>7,} filas')
print(f'macro_riesgo     : {len(df_riesgo):>7,} filas')

precios_empresas : 157,455 filas | 35 empresas
macro_act_mensual:     250 filas
macro_act_trimes :      82 filas
macro_mon_diario :   5,865 filas
macro_mon_mensual:     423 filas
macro_riesgo     :   5,091 filas


## 3. Alineación temporal al calendario bursátil

Creamos un **calendario bursátil** con todas las fechas de negociación únicas del IBEX 35. Luego usamos  para alinear cada tabla macro a frecuencia diaria mediante **forward-fill**.Con esta función cada día de trading recibe el último valor macro disponible hasta esa fecha.

Por ejemplo, el tipo de interés de enero se aplica a todos los días de enero hasta que cambie.

In [11]:
# Calendario bursátil: fechas únicas de negociación
calendario = pd.DataFrame({'fecha': sorted(df_precios['fecha'].unique())})
print(f'Días de negociación únicos: {len(calendario)}')
print(f'Rango: {calendario["fecha"].min().date()} → {calendario["fecha"].max().date()}')

  # Función de alineación con forward-fill interno
def alinear_macro(df_macro, calendario):
      df = df_macro.sort_values('fecha').ffill().copy()
      return pd.merge_asof(calendario, df, on='fecha', direction='backward')

  # Alinear cada tabla macro al calendario diario
cal_act_mens   = alinear_macro(df_act_mens,   calendario)
cal_act_trim   = alinear_macro(df_act_trim,   calendario)
cal_mon_diario = alinear_macro(df_mon_diario, calendario)
cal_mon_mens   = alinear_macro(df_mon_mens,   calendario)
cal_riesgo     = alinear_macro(df_riesgo,     calendario)

print('Tablas alineadas al calendario:')
for nombre, df in [('act_mens', cal_act_mens), ('act_trim', cal_act_trim),
                     ('mon_diario', cal_mon_diario), ('mon_mens', cal_mon_mens),
                     ('riesgo', cal_riesgo)]:
      print(f'  {nombre:<15}: {len(df)} filas')


Días de negociación únicos: 5323
Rango: 2005-01-03 → 2025-10-31
Tablas alineadas al calendario:
  act_mens       : 5323 filas
  act_trim       : 5323 filas
  mon_diario     : 5323 filas
  mon_mens       : 5323 filas
  riesgo         : 5323 filas


## 4. Construcción del dataset maestro

Unimos  con todas las variables macro alineadas. Uso la función `join`, se hace sobre cada empresa y en cada día de trading recibe las variables macro correspondientes a ese día.

In [12]:
# Consolidar todas las macro en una sola tabla diaria                                                                                                                                                 
df_macro_diario = calendario.copy()
for df_cal in [cal_mon_diario, cal_riesgo, cal_act_mens, cal_mon_mens, cal_act_trim]:
      cols_nuevas = [c for c in df_cal.columns if c != 'fecha']
      df_macro_diario = df_macro_diario.merge(df_cal[['fecha'] + cols_nuevas], on='fecha', how='left')

print(f'Tabla macro diaria consolidada: {df_macro_diario.shape}')
print(f'Columnas: {list(df_macro_diario.columns)}')

  # Unir con precios de empresas
df_master = df_precios.merge(df_macro_diario, on='fecha', how='left')
df_master = df_master.sort_values(['ticker', 'fecha']).reset_index(drop=True)

print(f'Dataset maestro: {df_master.shape[0]:,} filas x {df_master.shape[1]} columnas')
print(f'Período: {df_master["fecha"].min().date()} → {df_master["fecha"].max().date()}')
print(f'Empresas: {df_master["ticker"].nunique()}')


Tabla macro diaria consolidada: (5323, 21)
Columnas: ['fecha', 'bono_es_10y', 'bono_de_10y', 'spread_es_de', 'eur_usd', 'vix', 'vibex', 'vstoxx', 'brent', 'gas_ttf', 'ipi_yoy', 'pmi', 'ipc_sub_mom', 'euribor_3m', 'euribor_6m', 'tipo_dfr', 'tipo_mlf', 'tipo_mro', 'ipc_yoy', 'pib_yoy', 'tasa_paro']
Dataset maestro: 157,455 filas x 31 columnas
Período: 2005-01-03 → 2025-10-31
Empresas: 35


## 5. Control de calidad del dataset maestro

Para ver la calidad del nuevo dataset investigo la cantidad de nulos y su porcentaje y la cobertura de datos disponibles

In [13]:
print('=== CONTROL DE CALIDAD — DATASET MAESTRO ===')
print()

# Nulos por columna (en % sobre el total de filas)
nulos = df_master.isnull().sum()
nulos_pct = (nulos / len(df_master) * 100).round(1)
df_nulos = pd.DataFrame({'nulos': nulos, '%': nulos_pct})
df_nulos = df_nulos[df_nulos['nulos'] > 0].sort_values('%', ascending=False)
print('Columnas con valores nulos:')
print(df_nulos.to_string())
print()

# Duplicados
dupl = df_master.duplicated(subset=['ticker','fecha']).sum()
print(f'Filas duplicadas (ticker+fecha): {dupl}')
print()

# Cobertura de variables macro por fecha
print('Cobertura temporal de variables macro clave:')
for col in ['vix','euribor_3m','pib_yoy','ipc_yoy','bono_es_10y']:
    if col in df_master.columns:
        n = df_master[col].notna().sum()
        pct = n / len(df_master) * 100
        print(f'  {col:<15}: {n:>7,} ({pct:.1f}%)')

=== CONTROL DE CALIDAD — DATASET MAESTRO ===

Columnas con valores nulos:
               nulos     %
pmi           130958  83.2
vibex          95931  60.9
spread_es_de   74688  47.4
vstoxx         50613  32.1
ipc_yoy        45853  29.1
gas_ttf        30763  19.5
tipo_mro       21274  13.5
tipo_mlf       21274  13.5
tipo_dfr       21274  13.5
bono_es_10y    16282  10.3
bono_de_10y    16282  10.3
brent           7673   4.9
vix             5940   3.8
pib_yoy         1220   0.8
tasa_paro       1220   0.8
vol_hist_21d     350   0.2
euribor_3m       380   0.2
euribor_6m       380   0.2
pct_chg           43   0.0
log_ret           35   0.0
volume            18   0.0
high              18   0.0
low               19   0.0
open              12   0.0
net               43   0.0

Filas duplicadas (ticker+fecha): 0

Cobertura temporal de variables macro clave:
  vix            : 151,515 (96.2%)
  euribor_3m     : 157,075 (99.8%)
  pib_yoy        : 156,235 (99.2%)
  ipc_yoy        : 111,602 (70.9%)
  

## 6. Guardado del dataset maestro

Guardo la nueva tabla de `dataset maestro` dentro de `tfg.db` y el backup en csv y compruebo que se haya guardado correctamente 

In [16]:
# Guardar en SQLite                                                                                                                                                                                   
df_master.to_sql(
      name      = 'dataset_maestro',                                                                                                                                                                    
      con       = engine,
      if_exists = 'replace',
      index     = False,
      dtype     = {'fecha': Date()}
  )

  # Backup en CSV
csv_path = os.path.join(PROC_DIR, 'dataset_maestro.csv')
df_master.to_csv(csv_path, index=False)

print(f'dataset_maestro guardado en tfg.db: {len(df_master):,} filas x {df_master.shape[1]} columnas')
print(f'CSV backup: {csv_path}')

  # Verificación final
with sqlite3.connect(DB_PATH) as conn:
      tablas = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn)
      print('Tablas en tfg.db:')
      for t in tablas['name']:
          n = pd.read_sql(f'SELECT COUNT(*) as n FROM "{t}"', conn).iloc[0,0]
          print(f'  {t:<35} {n:>7,} filas')

dataset_maestro guardado en tfg.db: 157,455 filas x 31 columnas
CSV backup: /Users/adriancelada/Library/Mobile Documents/com~apple~CloudDocs/UFV/UNIVERSIDAD FRANCISCO DE VITORIA/4º/TFG/proyecto/data/processed/dataset_maestro.csv
Tablas en tfg.db:
  dataset_maestro                     157,455 filas
  macro_act_mensual                       250 filas
  macro_act_trimes                         82 filas
  macro_mon_diario                      5,865 filas
  macro_mon_mensual                       423 filas
  macro_riesgo                          5,091 filas
  precios_empresas                    157,455 filas
  ref_empresas                             36 filas
